# 05 协议感知模型训练（Protocol-Aware）

目标：先回答一个关键问题——**是否应针对不同协议训练不同模型**。

建议采用两阶段：
1. **阶段A（公平基线）**：同一个模型在三协议上全部跑一遍，建立统一参照。
2. **阶段B（协议感知）**：对每个协议在候选模型池内自动择优。

这样既保证方法学严谨，又能体现系统设计价值。

In [1]:
import os
import sys
import pandas as pd
import torch

PROJECT_ROOT = r"C:\\Users\\PLUTO\\Desktop\\battery-rul"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.rul_system.data import (
    load_battery_raw_parameters,
    default_condition_map,
    build_protocol_splits,
)
from src.rul_system.models import BaselineLSTM, ConditionAwareTransformer
from src.rul_system.train import run_protocol_experiment

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


## 1) 加载当前可用数据（NASA示例）

In [2]:
PROCESSED_ROOT = os.path.join(PROJECT_ROOT, "data\\processed")

condition_map = default_condition_map()
target_ids = set(condition_map.keys())

battery_dict = {}
group_load_stats = []

for group_name in sorted(os.listdir(PROCESSED_ROOT)):
    group_path = os.path.join(PROCESSED_ROOT, group_name)
    if not os.path.isdir(group_path):
        continue

    pkl_files = [f for f in os.listdir(group_path) if f.endswith(".pkl")]
    group_ids = [os.path.splitext(f)[0] for f in pkl_files]
    group_ids = [bid for bid in group_ids if bid in target_ids]

    if not group_ids:
        continue

    group_data = load_battery_raw_parameters(group_path, group_ids)
    battery_dict.update(group_data)
    group_load_stats.append((group_name, len(group_ids), sorted(group_data.keys())))

available_ids = sorted([bid for bid in battery_dict if bid in condition_map])

print("=== 分组加载统计 ===")
for group_name, num_ids, loaded_ids in group_load_stats:
    print(f"{group_name}: {num_ids} cells -> {loaded_ids}")

print("\n总加载电池数:", len(available_ids))
print("available_ids:", available_ids)

=== 分组加载统计 ===
1. BatteryAgingARC-FY08Q4: 4 cells -> ['B0005', 'B0006', 'B0007', 'B0018']
2. BatteryAgingARC_25_26_27_28_P1: 4 cells -> ['B0025', 'B0026', 'B0027', 'B0028']
3. BatteryAgingARC_25-44: 18 cells -> ['B0025', 'B0026', 'B0027', 'B0028', 'B0029', 'B0030', 'B0031', 'B0032', 'B0033', 'B0034', 'B0036', 'B0038', 'B0039', 'B0040', 'B0041', 'B0042', 'B0043', 'B0044']
4. BatteryAgingARC_45_46_47_48: 4 cells -> ['B0045', 'B0046', 'B0047', 'B0048']
5. BatteryAgingARC_49_50_51_52: 4 cells -> ['B0049', 'B0050', 'B0051', 'B0052']
6. BatteryAgingARC_53_54_55_56: 4 cells -> ['B0053', 'B0054', 'B0055', 'B0056']

总加载电池数: 34
available_ids: ['B0005', 'B0006', 'B0007', 'B0018', 'B0025', 'B0026', 'B0027', 'B0028', 'B0029', 'B0030', 'B0031', 'B0032', 'B0033', 'B0034', 'B0036', 'B0038', 'B0039', 'B0040', 'B0041', 'B0042', 'B0043', 'B0044', 'B0045', 'B0046', 'B0047', 'B0048', 'B0049', 'B0050', 'B0051', 'B0052', 'B0053', 'B0054', 'B0055', 'B0056']


## 2) 构建三协议划分

In [3]:
protocols = build_protocol_splits(available_ids, condition_map)
for protocol_name in ["same_condition", "lobo", "loco"]:
    print(protocol_name, "splits:", len(protocols[protocol_name]))

same_condition splits: 34
lobo splits: 34
loco splits: 6


### 三协议回顾（便于写论文时直接引用）

- `same_condition`：同一工况组内留一电池测试（同分布泛化）。
- `lobo`（Leave-One-Battery-Out）：留一电池测试，其余电池训练（跨个体泛化）。
- `loco`（Leave-One-Condition-Out）：留一整组工况测试，其余工况训练（跨工况泛化，难度最高）。

## 3) 协议感知模型池（第一版）

说明：这里先用现有可运行模型作为模型池。
后续你可把 `Attention-LSTM-GRU` 加入对应协议候选中。

In [4]:
SEQ_LENGTH = 10
BATCH_SIZE = 32
TRAIN_KWARGS = {"epochs": 20, "lr": 1e-3}

MODEL_ZOO = {
    "LSTM": (BaselineLSTM, {"input_dim": 5, "hidden_dim": 64, "num_layers": 2, "dropout": 0.1}),
    "Transformer": (ConditionAwareTransformer, {"input_dim": 5, "d_model": 64, "nhead": 4, "num_layers": 2, "dim_feedforward": 128, "dropout": 0.1, "max_seq_len": 128}),
}

PROTOCOL_MODEL_CANDIDATES = {
    "same_condition": ["LSTM", "Transformer"],
    "lobo": ["LSTM", "Transformer"],
    "loco": ["LSTM", "Transformer"],
}

In [5]:
def run_protocol_model_search(protocol_name, protocol_splits, max_splits=1):
    """
    在一个协议上，对候选模型进行对比搜索。
    max_splits=1 用于快速冒烟；后续可改为 None 或更大值。
    """
    splits = protocol_splits if max_splits is None else protocol_splits[:max_splits]
    all_records = []

    for model_name in PROTOCOL_MODEL_CANDIDATES[protocol_name]:
        model_cls, model_kwargs = MODEL_ZOO[model_name]
        records = run_protocol_experiment(
            battery_dict=battery_dict,
            condition_map=condition_map,
            protocol_splits=splits,
            model_cls=model_cls,
            model_kwargs=model_kwargs,
            train_kwargs=TRAIN_KWARGS,
            seq_length=SEQ_LENGTH,
            batch_size=BATCH_SIZE,
            device=device,
        )
        for r in records:
            r["protocol"] = protocol_name
            r["model"] = model_name
        all_records.extend(records)

    return pd.DataFrame(all_records)

In [6]:
all_protocol_results = []
for protocol_name in ["same_condition", "lobo", "loco"]:
    if len(protocols[protocol_name]) == 0:
        continue
    df_part = run_protocol_model_search(protocol_name, protocols[protocol_name], max_splits=1)
    all_protocol_results.append(df_part)

df_search = pd.concat(all_protocol_results, ignore_index=True) if all_protocol_results else pd.DataFrame()
display(df_search)

if not df_search.empty:
    summary = (
        df_search.groupby(["protocol", "model"])[["RMSE", "MAE", "MAPE"]]
        .mean()
        .reset_index()
    )
    print("\n=== 协议-模型对比（首轮冒烟）===")
    display(summary)

    best_by_protocol = summary.sort_values(["protocol", "MAE"]).groupby("protocol").head(1)
    print("\n=== 每个协议当前最优模型（按 MAE）===")
    display(best_by_protocol)

,protocol_group,train_ids,test_id,RMSE,MAE,MAPE,num_points,protocol,model
0,T24_I2,"[B0006, B0007, B0018, B0036]",B0005,0.080989,0.075142,5.064054,158,same_condition,LSTM
1,T24_I2,"[B0006, B0007, B0018, B0036]",B0005,0.039718,0.036216,2.417083,158,same_condition,Transformer
2,mixed,"[B0006, B0007, B0018, B0025, B0026, B0027, B00...",B0005,0.207159,0.173606,12.138416,158,lobo,LSTM
3,mixed,"[B0006, B0007, B0018, B0025, B0026, B0027, B00...",B0005,0.222610,0.199317,13.730370,158,lobo,Transformer
4,T24_I2,"[B0025, B0026, B0027, B0028, B0029, B0030, B00...",B0005,0.276530,0.225195,15.894951,158,loco,LSTM
5,T24_I2,"[B0025, B0026, B0027, B0028, B0029, B0030, B00...",B0006,0.225091,0.175593,10.934746,158,loco,LSTM
6,T24_I2,"[B0025, B0026, B0027, B0028, B0029, B0030, B00...",B0007,0.206243,0.159820,9.158387,158,loco,LSTM
7,T24_I2,"[B0025, B0026, B0027, B0028, B0029, B0030, B00...",B0018,0.124169,0.104641,6.614786,122,loco,LSTM
8,T24_I2,"[B0025, B0026, B0027, B0028, B0029, B0030, B00...",B0036,0.208495,0.189623,10.978844,187,loco,LSTM
9,T24_I2,"[B0025, B0026, B0027, B0028, B0029, B0030, B00...",B0005,0.172076,0.154031,10.353134,158,loco,Transformer



=== 协议-模型对比（首轮冒烟）===


,protocol,model,RMSE,MAE,MAPE
0,lobo,LSTM,0.207159,0.173606,12.138416
1,lobo,Transformer,0.222610,0.199317,13.730370
2,loco,LSTM,0.208106,0.170974,10.716343
3,loco,Transformer,0.191213,0.166639,10.485568
4,same_condition,LSTM,0.080989,0.075142,5.064054
5,same_condition,Transformer,0.039718,0.036216,2.417083



=== 每个协议当前最优模型（按 MAE）===


,protocol,model,RMSE,MAE,MAPE
0,lobo,LSTM,0.207159,0.173606,12.138416
3,loco,Transformer,0.191213,0.166639,10.485568
5,same_condition,Transformer,0.039718,0.036216,2.417083


## 4) 下一步

1. 把 `max_splits=1` 改为更大值或全量，获得稳定统计结果。
2. 将 `Attention-LSTM-GRU` 加入 `MODEL_ZOO`。
3. 扩展到 CALCE 等新数据集，做跨数据集验证。

In [8]:
print("=== 精简汇总：数据规模与协议 ===")
print("电池总数:", len(available_ids))
print("same/lobo/loco splits:", len(protocols["same_condition"]), len(protocols["lobo"]), len(protocols["loco"]))

print("\n=== 精简汇总：模型搜索结果 ===")
if 'df_search' in globals() and not df_search.empty:
    print("记录条数:", len(df_search))
    print(df_search[["protocol", "model", "test_id", "MAE", "RMSE", "MAPE", "num_points"]].head(12).to_string(index=False))
else:
    print("df_search 为空")

if 'summary' in globals() and not summary.empty:
    print("\n协议-模型均值(按MAE升序):")
    print(summary.sort_values(["protocol", "MAE"]).to_string(index=False))
else:
    print("summary 为空")

if 'best_by_protocol' in globals() and not best_by_protocol.empty:
    print("\n每协议最佳模型:")
    print(best_by_protocol.to_string(index=False))
else:
    print("best_by_protocol 为空")

=== 精简汇总：数据规模与协议 ===
电池总数: 34
same/lobo/loco splits: 34 34 6

=== 精简汇总：模型搜索结果 ===
记录条数: 14
      protocol       model test_id      MAE     RMSE      MAPE  num_points
same_condition        LSTM   B0005 0.075142 0.080989  5.064054         158
same_condition Transformer   B0005 0.036216 0.039718  2.417083         158
          lobo        LSTM   B0005 0.173606 0.207159 12.138416         158
          lobo Transformer   B0005 0.199317 0.222610 13.730370         158
          loco        LSTM   B0005 0.225195 0.276530 15.894951         158
          loco        LSTM   B0006 0.175593 0.225091 10.934746         158
          loco        LSTM   B0007 0.159820 0.206243  9.158387         158
          loco        LSTM   B0018 0.104641 0.124169  6.614786         122
          loco        LSTM   B0036 0.189623 0.208495 10.978844         187
          loco Transformer   B0005 0.154031 0.172076 10.353134         158
          loco Transformer   B0006 0.205163 0.232953 13.928469         158
         